# Prior Rollout Module Unit Tests

This notebook tests the functionality of the `prior_rollout.py` module which provides
prior-only rollout evaluation during PPO training.

## 1. Setup and Imports

In [1]:
import os
import sys

# Add the track-mjx directory to path
sys.path.insert(0, '/home/mila/a/aidan.sirbu/track-mjx')
sys.path.insert(0, '/home/mila/a/aidan.sirbu/vnl-playground')

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"

import jax
import jax.numpy as jnp
from jax import random
import numpy as np
from brax.training.acme import running_statistics
from brax.training.acme import specs

print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")

JAX version: 0.7.2
Available devices: [CudaDevice(id=0)]


In [2]:
# Import the prior_rollout module
from track_mjx.agent.mlp_ppo import prior_rollout
from track_mjx.agent.mlp_ppo import intention_network

print("Prior rollout module imported successfully!")

Prior rollout module imported successfully!


## 2. Test `check_termination` Function

This function checks if a state should be terminated based on:
1. NaN detection in the physics state
2. Torso height outside healthy range

In [3]:
# Create a mock data object for testing
from dataclasses import dataclass
from typing import Any

@dataclass
class MockData:
    """Mock MJX Data object for testing."""
    qpos: jnp.ndarray
    qvel: jnp.ndarray


def test_check_termination():
    """Test the check_termination function."""
    healthy_z_range = (0.0325, 0.5)
    
    # Test 1: Normal state within range - should NOT terminate
    normal_qpos = jnp.array([0.0, 0.0, 0.2, 1.0, 0.0, 0.0, 0.0])  # z=0.2 is within range
    normal_qvel = jnp.zeros(6)
    normal_data = MockData(qpos=normal_qpos, qvel=normal_qvel)
    
    result = prior_rollout.check_termination(normal_data, healthy_z_range)
    assert not result, f"Expected False for normal state, got {result}"
    print("✓ Test 1 passed: Normal state within range - no termination")
    
    # Test 2: State with NaN - should terminate
    nan_qpos = jnp.array([0.0, 0.0, float('nan'), 1.0, 0.0, 0.0, 0.0])
    nan_data = MockData(qpos=nan_qpos, qvel=normal_qvel)
    
    result = prior_rollout.check_termination(nan_data, healthy_z_range)
    assert result, f"Expected True for NaN state, got {result}"
    print("✓ Test 2 passed: NaN state - termination triggered")
    
    # Test 3: Z position too low - should terminate
    low_z_qpos = jnp.array([0.0, 0.0, 0.01, 1.0, 0.0, 0.0, 0.0])  # z=0.01 is below min
    low_z_data = MockData(qpos=low_z_qpos, qvel=normal_qvel)
    
    result = prior_rollout.check_termination(low_z_data, healthy_z_range)
    assert result, f"Expected True for low z position, got {result}"
    print("✓ Test 3 passed: Low z position - termination triggered")
    
    # Test 4: Z position too high - should terminate
    high_z_qpos = jnp.array([0.0, 0.0, 0.7, 1.0, 0.0, 0.0, 0.0])  # z=0.7 is above max
    high_z_data = MockData(qpos=high_z_qpos, qvel=normal_qvel)
    
    result = prior_rollout.check_termination(high_z_data, healthy_z_range)
    assert result, f"Expected True for high z position, got {result}"
    print("✓ Test 4 passed: High z position - termination triggered")
    
    # Test 5: NaN in qvel - should terminate
    nan_qvel = jnp.array([0.0, float('nan'), 0.0, 0.0, 0.0, 0.0])
    nan_qvel_data = MockData(qpos=normal_qpos, qvel=nan_qvel)
    
    result = prior_rollout.check_termination(nan_qvel_data, healthy_z_range)
    assert result, f"Expected True for NaN in qvel, got {result}"
    print("✓ Test 5 passed: NaN in qvel - termination triggered")
    
    print("\n✓ All check_termination tests passed!")


test_check_termination()

✓ Test 1 passed: Normal state within range - no termination
✓ Test 2 passed: NaN state - termination triggered
✓ Test 3 passed: Low z position - termination triggered
✓ Test 4 passed: High z position - termination triggered
✓ Test 5 passed: NaN in qvel - termination triggered

✓ All check_termination tests passed!


## 3. Test `extract_prior_decoder_params` Function

This function extracts prior and decoder parameters from the full intention network policy params.

In [4]:
def test_extract_prior_decoder_params():
    """Test the extract_prior_decoder_params function."""
    # Create mock policy parameters
    mock_normalizer_params = running_statistics.RunningStatisticsState(
        count=jnp.array(100.0),
        mean=jnp.zeros(100),
        summed_variance=jnp.ones(100),
        std=jnp.ones(100),
    )
    
    mock_network_params = {
        "params": {
            "encoder": {"layer1": jnp.ones((10, 10))},
            "prior": {"layer1": jnp.ones((10, 10)) * 2},
            "decoder": {"layer1": jnp.ones((10, 10)) * 3},
        }
    }
    
    policy_params = (mock_normalizer_params, mock_network_params)
    
    # Extract params
    prior_params, decoder_params, normalizer_params = prior_rollout.extract_prior_decoder_params(policy_params)
    
    # Check that prior params are correctly extracted
    assert "layer1" in prior_params, "Prior params should have 'layer1'"
    assert jnp.allclose(prior_params["layer1"], jnp.ones((10, 10)) * 2), "Prior params values incorrect"
    print("✓ Test 1 passed: Prior params correctly extracted")
    
    # Check that decoder params are correctly extracted
    assert "layer1" in decoder_params, "Decoder params should have 'layer1'"
    assert jnp.allclose(decoder_params["layer1"], jnp.ones((10, 10)) * 3), "Decoder params values incorrect"
    print("✓ Test 2 passed: Decoder params correctly extracted")
    
    # Check that normalizer params are passed through
    assert normalizer_params.count == mock_normalizer_params.count, "Normalizer params count mismatch"
    print("✓ Test 3 passed: Normalizer params correctly passed through")
    
    print("\n✓ All extract_prior_decoder_params tests passed!")


test_extract_prior_decoder_params()

✓ Test 1 passed: Prior params correctly extracted
✓ Test 2 passed: Decoder params correctly extracted
✓ Test 3 passed: Normalizer params correctly passed through

✓ All extract_prior_decoder_params tests passed!


## 4. Test `reparameterize` Function

This function samples from a Gaussian distribution using the reparameterization trick.

In [6]:
def test_reparameterize():
    """Test the reparameterize function."""
    key = random.PRNGKey(0)
    
    # Test 1: Output shape matches input shape
    mean = jnp.zeros((10, 5))
    logvar = jnp.zeros((10, 5))  # variance = 1
    
    z = prior_rollout.reparameterize(key, mean, logvar)
    assert z.shape == mean.shape, f"Expected shape {mean.shape}, got {z.shape}"
    print("✓ Test 1 passed: Output shape matches input shape")
    
    # Test 2: With very small variance (logvar = -40), output approximately equals mean
    # exp(-40/2) = exp(-20) ≈ 2e-9 std, so noise contribution is negligible
    logvar_zero = jnp.full((10, 5), -40.0)
    z_deterministic = prior_rollout.reparameterize(key, mean, logvar_zero)
    assert jnp.allclose(z_deterministic, mean, atol=1e-5), f"With near-zero variance, output should equal mean. Got max diff: {jnp.max(jnp.abs(z_deterministic - mean))}"
    print("✓ Test 2 passed: Near-zero variance produces deterministic output")
    
    # Test 3: Different keys produce different samples
    key1 = random.PRNGKey(0)
    key2 = random.PRNGKey(1)
    logvar_normal = jnp.zeros((10, 5))
    
    z1 = prior_rollout.reparameterize(key1, mean, logvar_normal)
    z2 = prior_rollout.reparameterize(key2, mean, logvar_normal)
    assert not jnp.allclose(z1, z2), "Different keys should produce different samples"
    print("✓ Test 3 passed: Different keys produce different samples")
    
    # Test 4: Samples are centered around mean (statistical test)
    mean_nonzero = jnp.ones((1000, 5)) * 5.0
    logvar_small = jnp.full((1000, 5), -2.0)  # small variance
    
    z_samples = prior_rollout.reparameterize(key, mean_nonzero, logvar_small)
    sample_mean = jnp.mean(z_samples, axis=0)
    assert jnp.allclose(sample_mean, 5.0, atol=0.5), f"Sample mean {sample_mean} should be close to 5.0"
    print("✓ Test 4 passed: Samples are centered around the mean")
    
    print("\n✓ All reparameterize tests passed!")


test_reparameterize()

✓ Test 1 passed: Output shape matches input shape
✓ Test 2 passed: Near-zero variance produces deterministic output
✓ Test 3 passed: Different keys produce different samples
✓ Test 4 passed: Samples are centered around the mean

✓ All reparameterize tests passed!


## 5. Test Prior and Decoder Network Modules

Test that the Prior and Decoder modules from intention_network work correctly.

In [7]:
def test_prior_decoder_modules():
    """Test the Prior and Decoder network modules."""
    key = random.PRNGKey(42)
    
    # Test Prior module
    prior_layer_sizes = [256, 128]
    latent_size = 60
    input_size = 100
    
    prior = intention_network.Prior(
        layer_sizes=prior_layer_sizes,
        latents=latent_size,
    )
    
    # Initialize and apply
    dummy_input = jnp.zeros((1, input_size))
    prior_params = prior.init(key, dummy_input)
    prior_mean, prior_logvar = prior.apply(prior_params, dummy_input)
    
    assert prior_mean.shape == (1, latent_size), f"Expected shape (1, {latent_size}), got {prior_mean.shape}"
    assert prior_logvar.shape == (1, latent_size), f"Expected shape (1, {latent_size}), got {prior_logvar.shape}"
    print("✓ Prior module produces correct output shapes")
    
    # Test Decoder module
    decoder_layer_sizes = [256, 128]
    action_param_size = 64  # e.g., 32 actions * 2 (mean + std)
    decoder_input_size = latent_size + input_size
    
    decoder = intention_network.Decoder(
        layer_sizes=decoder_layer_sizes + [action_param_size],
    )
    
    # Initialize and apply
    key, subkey = random.split(key)
    decoder_input = jnp.zeros((1, decoder_input_size))
    decoder_params = decoder.init(subkey, decoder_input)
    action_logits, _ = decoder.apply(decoder_params, decoder_input)
    
    assert action_logits.shape == (1, action_param_size), f"Expected shape (1, {action_param_size}), got {action_logits.shape}"
    print("✓ Decoder module produces correct output shapes")
    
    print("\n✓ All Prior/Decoder module tests passed!")


test_prior_decoder_modules()

✓ Prior module produces correct output shapes
✓ Decoder module produces correct output shapes

✓ All Prior/Decoder module tests passed!


## 6. Test `create_prior_policy` Function

Test that the policy function created from prior and decoder params works correctly.

In [8]:
def test_create_prior_policy():
    """Test the create_prior_policy function."""
    key = random.PRNGKey(42)
    
    # Setup dimensions
    proprioceptive_obs_size = 100
    latent_size = 60
    action_size = 32
    prior_layer_sizes = [256, 128]
    decoder_layer_sizes = [256, 128]
    
    # Create mock normalizer params (identity normalization)
    normalizer_params = running_statistics.RunningStatisticsState(
        count=jnp.array(100.0),
        mean=jnp.zeros(proprioceptive_obs_size),
        summed_variance=jnp.ones(proprioceptive_obs_size) * 100,
        std=jnp.ones(proprioceptive_obs_size),
    )
    
    # Initialize Prior network
    prior = intention_network.Prior(
        layer_sizes=prior_layer_sizes,
        latents=latent_size,
    )
    key, prior_key = random.split(key)
    prior_params = prior.init(prior_key, jnp.zeros((1, proprioceptive_obs_size)))
    
    # Initialize Decoder network
    from brax.training import distribution
    parametric_action_distribution = distribution.NormalTanhDistribution(event_size=action_size)
    action_param_size = parametric_action_distribution.param_size
    
    decoder = intention_network.Decoder(
        layer_sizes=decoder_layer_sizes + [action_param_size],
    )
    key, decoder_key = random.split(key)
    decoder_input_size = latent_size + proprioceptive_obs_size
    decoder_params = decoder.init(decoder_key, jnp.zeros((1, decoder_input_size)))
    
    # Create policy function
    policy_fn = prior_rollout.create_prior_policy(
        prior_network_params=prior_params["params"],
        decoder_network_params=decoder_params["params"],
        normalizer_params=normalizer_params,
        intention_latent_size=latent_size,
        action_size=action_size,
        proprioceptive_obs_size=proprioceptive_obs_size,
        decoder_hidden_layer_sizes=decoder_layer_sizes,
        prior_hidden_layer_sizes=prior_layer_sizes,
        preprocess_observations_fn=running_statistics.normalize,
        fixed_logvar=-2.0,
        deterministic=False,
    )
    
    # Test policy function
    key, action_key = random.split(key)
    obs = jnp.zeros((1, proprioceptive_obs_size))
    
    action, extras = policy_fn(obs, action_key)
    
    # Check action shape
    assert action.shape == (1, action_size), f"Expected action shape (1, {action_size}), got {action.shape}"
    print("✓ Policy function produces correct action shape")
    
    # Check extras contain expected keys
    expected_keys = ["prior_mean", "prior_logvar", "intention", "logits"]
    for key_name in expected_keys:
        assert key_name in extras, f"Expected '{key_name}' in extras"
    print("✓ Policy function returns expected extras")
    
    # Check that action is bounded (tanh output)
    assert jnp.all(action >= -1.0) and jnp.all(action <= 1.0), "Actions should be bounded by tanh"
    print("✓ Actions are bounded within [-1, 1]")
    
    # Test deterministic mode
    policy_fn_det = prior_rollout.create_prior_policy(
        prior_network_params=prior_params["params"],
        decoder_network_params=decoder_params["params"],
        normalizer_params=normalizer_params,
        intention_latent_size=latent_size,
        action_size=action_size,
        proprioceptive_obs_size=proprioceptive_obs_size,
        decoder_hidden_layer_sizes=decoder_layer_sizes,
        prior_hidden_layer_sizes=prior_layer_sizes,
        preprocess_observations_fn=running_statistics.normalize,
        fixed_logvar=-2.0,
        deterministic=True,
    )
    
    key1, key2 = random.split(key, 2)
    action1, _ = policy_fn_det(obs, key1)
    action2, _ = policy_fn_det(obs, key2)
    
    assert jnp.allclose(action1, action2), "Deterministic policy should produce same actions"
    print("✓ Deterministic policy produces consistent actions")
    
    print("\n✓ All create_prior_policy tests passed!")


test_create_prior_policy()

✓ Policy function produces correct action shape
✓ Policy function returns expected extras
✓ Actions are bounded within [-1, 1]
✓ Deterministic policy produces consistent actions

✓ All create_prior_policy tests passed!


## 7. Integration Test with Real Environment (Optional)

This test uses the real VNL Imitation environment to test the full prior rollout evaluation.

In [ ]:
# This cell requires the VNL environment and reference data to be available
# Uncomment and run if you want to test with the real environment

# try:
#     from vnl_playground.tasks.rodent import imitation
#     from vnl_playground.tasks.rodent import wrappers as vnl_wrappers
#     from vnl_playground.tasks.rodent import consts as rodent_consts
#     from vnl_playground.tasks.rodent.reference_clips import ReferenceClips
#     
#     print("VNL environment modules imported successfully!")
#     HAS_VNL_ENV = True
# except ImportError as e:
#     print(f"Could not import VNL environment: {e}")
#     HAS_VNL_ENV = False

print("Integration test with real environment skipped (requires reference data).")
print("To run the integration test, uncomment the cells below and ensure reference data is available.")

In [ ]:
# def test_with_real_environment():
#     """Test prior rollout evaluation with real VNL environment."""
#     if not HAS_VNL_ENV:
#         print("Skipping real environment test - VNL not available")
#         return
#     
#     # Create environment config
#     env_cfg = imitation.default_config()
#     env_cfg.clip_length = 250
#     env_cfg.start_frame_range = [0, 50]
#     
#     # Load reference clips
#     reference_clips = ReferenceClips(
#         data_path=rodent_consts.IMITATION_REFERENCE_PATH,
#         n_frames_per_clip=env_cfg.clip_length,
#     )
#     
#     # Create environment
#     env = imitation.Imitation(config=env_cfg, clips=reference_clips)
#     
#     # Create mock policy parameters
#     proprioceptive_obs_size = env.proprioceptive_obs_size
#     action_size = env.action_size
#     latent_size = 60
#     
#     # ... rest of integration test
#     print("✓ Real environment test completed!")
# 
# test_with_real_environment()

## 8. Test Summary

Summary of all tests run in this notebook.

In [ ]:
print("="*60)
print("Test Summary")
print("="*60)
print()
print("✓ check_termination: Tests for NaN detection and z-range checking")
print("✓ extract_prior_decoder_params: Tests for parameter extraction")
print("✓ reparameterize: Tests for Gaussian sampling with reparameterization trick")
print("✓ Prior/Decoder modules: Tests for network module shapes")
print("✓ create_prior_policy: Tests for policy function creation and inference")
print()
print("All unit tests passed!")
print("="*60)